# Pydantic - Wprowadzenie

## Spis treści:
1. Co to Pydantic i po co?
2. BaseModel - podstawy
3. Field - walidacja
4. Typy i Optional
5. Zagnieżdżone modele
6. Serializacja - model_validate i model_dump
7. ValidationError
8. Podsumowanie

---

## 1. Co to Pydantic i po co?

**Pydantic** = biblioteka do **walidacji danych w runtime** używająca type hints.

### Problem bez Pydantic:

Type hints **NIE WALIDUJĄ** w runtime!

In [1]:
# Zwykły Python - type hints NIE chronią przed błędami!
class User:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

# Python pozwoli na błędne typy:
user = User(name=123, age="not a number")  # ✅ Python nie reaguje!
print(f"name={user.name} (type: {type(user.name).__name__})")
print(f"age={user.age} (type: {type(user.age).__name__})")
print("\n⚠️ Błędne typy, ale Python nie zgłasza błędu!")

name=123 (type: int)
age=not a number (type: str)

⚠️ Błędne typy, ale Python nie zgłasza błędu!


W jaki sposób moglibyśm walidować te wartości w runtimie?

In [2]:
class User:
    def __init__(self, name: str, age: int):
        if not isinstance(name, str):
            raise Exception(f"Wrong type of the `name` variable, should be string, got {type(name)}")
        if not isinstance(age, int):
            raise Exception(f"Wrong type of the `age` variable, should be int, got {type(name)}")
        self.name = name
        self.age = age

user = User(name=123, age="not a number")

Exception: Wrong type of the `name` variable, should be string, got <class 'int'>

A czy istnieje bardziej elegancki sposób?

### Rozwiązanie: Pydantic BaseModel

**Pydantic WALIDUJE typy w runtime wykorzystując do tego type hints**

In [3]:
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    name: str
    age: int

# ✅ Poprawne dane
user = User(name="Jan", age=25)
print(user)
print(f"name={user.name}, age={user.age}")

# ❌ Błędne dane - ValidationError!
try:
    bad_user = User(name=123, age="not a number")
except ValidationError as e:
    print(f"\n❌ ValidationError:\n{e}")

name='Jan' age=25
name=Jan, age=25

❌ ValidationError:
2 validation errors for User
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not a number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


In [4]:
# Spójrzmy jeszcze jak wygląda "czysty" bład walidacji.

bad_user = User(name=123, age="not a number")  # ValidationError

ValidationError: 2 validation errors for User
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not a number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

### Po co Pydantic?

1. **Walidacja danych** - rzuca ValidationError dla błędnych danych
2. **Automatyczna konwersja** - próbuje skonwertować typy ("25" → 25)
3. **Serializacja** - świetny serializer do/z dict i JSON (model ↔ dict ↔ JSON)
4. **FastAPI używa Pydantic** - do walidacji requestów i generowania dokumentacji
5. **Type safety** - mypy + runtime validation = podwójna ochrona

**Pydantic ≠ tylko FastAPI!** Używany wszędzie gdzie potrzebna walidacja (API, config, parsowanie JSON, etc.)

---

## 2. BaseModel - podstawy

### Definiowanie modelu

Model = klasa dziedzicząca po `BaseModel` z type hints:

In [6]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    email: str
    is_active: bool = True  # Wartość domyślna

# Tworzenie instancji
user = User(id=1, name="Jan", email="jan@example.com")
print(user)

# Dostęp do pól
print(f"\nname: {user.name}")
print(f"is_active: {user.is_active}")

# Bez is_active - użyje domyślnej wartości
user2 = User(id=2, name="Anna", email="anna@example.com")
print(f"\nuser2.is_active: {user2.is_active}")

id=1 name='Jan' email='jan@example.com' is_active=True

name: Jan
is_active: True

user2.is_active: True


### Co robi Pydantic pod maską?

Deklaracje `id: int`, `name: str` itp. **NIE SĄ atrybutami klasowymi!**

To **SCHEMAT** z którego Pydantic:
1. Generuje `__init__(id, name, email, is_active=True)` z walidacją typów
2. Tworzy **atrybuty INSTANCJI** (`self.id`, `self.name`, `self.email`, `self.is_active`)

Każdy obiekt ma **własne** atrybuty:
- `user.name` = "Jan" (instancja 1)
- `user2.name` = "Anna" (instancja 2)

**Pydantic robi dokładnie to samo co manualna klasa z walidacją `isinstance()` powyżej**, tylko automatycznie!

### Automatyczna konwersja typów

Pydantic **próbuje** skonwertować dane do odpowiedniego typu:

In [8]:
# String "25" zostanie skonwertowany na int 25
user = User(id="1", name="Jan", email="jan@example.com", is_active="yes")
print(user)
print(f"\nid type: {type(user.id)} (było str, jest int)")
print(f"is_active type: {type(user.is_active)} (było str, jest bool)")

# Ale jak konwersja niemożliwa - ValidationError
try:
    bad = User(id="abc", name="Jan", email="jan@example.com")
except ValidationError as e:
    print(f"\n❌ Nie można skonwertować 'abc' na int:\n{e}")

id=1 name='Jan' email='jan@example.com' is_active=True

id type: <class 'int'> (było str, jest int)
is_active type: <class 'bool'> (było str, jest bool)

❌ Nie można skonwertować 'abc' na int:
1 validation error for User
id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='abc', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


### Tworzenie z dict - rozpakowanie `**`

Możesz stworzyć model z dict używając operatora rozpakowania `**`:

**UWAGA:** To działa dla każdego modelu! To cecha języka python, a nie samego pydantic.
- `User(**data)` - rozpakowanie dict jako argumenty (działa dla każdego modelu)

W odróżnieniu od zagnieżdżonych modeli (sekcja 5), gdzie mamy składnie rozpakowywania dictów specyficzną wyłącznie dla pydantic:
- `User(address={"street": "..."})` - Pydantic konwertuje dict → model (tylko dla pól typu BaseModel)

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    email: str

# Dane z zewnątrz (np. z API, z pliku, z bazy danych)
user_data = {
    "id": 1,
    "name": "Jan",
    "email": "jan@example.com"
}

# Rozpakowanie dict jako argumenty
user = User(**user_data)
print(user)

# To jest równoważne z:
# user = User(id=1, name="Jan", email="jan@example.com")

# Przydatne gdy:
# - masz dane w dict (np. z JSON)
# - nie chcesz wypisywać wszystkich pól ręcznie
# - dane przychodzą dynamicznie

---

## 3. Field - walidacja

**Field** dodaje **reguły walidacji** (min/max długość, zakres, pattern, etc.):

In [9]:
from pydantic import BaseModel, Field

class User(BaseModel):
    id: int = Field(gt=0)  # greater than 0
    name: str = Field(min_length=3, max_length=50)
    age: int = Field(ge=0, le=120)  # 0 <= age <= 120
    email: str = Field(pattern=r".+@.+\..+")  # regex dla email (zapamiętaj, bo to się pojawi w ćwiczeniu)

# ✅ Poprawne dane
user = User(id=1, name="Jan", age=25, email="jan@example.com")
print(user)

# ❌ Błędy walidacji
print("\n--- Testowanie błędów walidacji ---")

try:
    User(id=0, name="Jan", age=25, email="jan@example.com")  # id musi być > 0
except ValidationError as e:
    print(f"id=0: {e.errors()[0]['msg']}")

try:
    User(id=1, name="Jo", age=25, email="jan@example.com")  # name za krótkie
except ValidationError as e:
    print(f"name='Jo': {e.errors()[0]['msg']}")

try:
    User(id=1, name="Jan", age=150, email="jan@example.com")  # age > 120
except ValidationError as e:
    print(f"age=150: {e.errors()[0]['msg']}")

try:
    User(id=1, name="Jan", age=25, email="invalid")  # zły format email
except ValidationError as e:
    print(f"email='invalid': {e.errors()[0]['msg']}")

id=1 name='Jan' age=25 email='jan@example.com'

--- Testowanie błędów walidacji ---
id=0: Input should be greater than 0
name='Jo': String should have at least 3 characters
age=150: Input should be less than or equal to 120
email='invalid': String should match pattern '.+@.+\..+'


### Najczęstsze Field constraints:

| Constraint | Opis | Przykład |
|------------|------|----------|
| `gt` | Greater than (>) | `Field(gt=0)` - większe od 0 |
| `ge` | Greater or equal (>=) | `Field(ge=0)` - >= 0 |
| `lt` | Less than (<) | `Field(lt=100)` - mniejsze od 100 |
| `le` | Less or equal (<=) | `Field(le=100)` - <= 100 |
| `min_length` | Min długość string/list | `Field(min_length=3)` |
| `max_length` | Max długość string/list | `Field(max_length=50)` |
| `pattern` | Regex pattern | `Field(pattern=r"^[A-Z]")` |
| `default` | Wartość domyślna | `Field(default=True)` |
| `description` | Opis (dla docs) | `Field(description="User ID")` |

---

## 4. Typy i Optional

### Podstawowe typy

In [ ]:
from pydantic import BaseModel

class Example(BaseModel):
    # Podstawowe
    text: str
    number: int
    price: float
    active: bool
    
    # Kolekcje
    tags: list[str]
    scores: dict[str, int]
    coordinates: tuple[float, float]

example = Example(
    text="Hello",
    number=42,
    price=19.99,
    active=True,
    tags=["python", "pydantic"],
    scores={"math": 90, "english": 85},
    coordinates=(52.2297, 21.0122)
)

print(example)

### Optional - wartość lub None

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    middle_name: str | None = None  # Opcjonalne (może być None)
    age: int | None = None

# Bez opcjonalnych pól
user1 = User(name="Jan")
print(user1)

# Z opcjonalnymi polami
user2 = User(name="Jan", middle_name="Kowalski", age=25)
print(user2)

**UWAGA:** `str | None = None` ≠ `str` z wartością domyślną!

```python
# Opcjonalne - może być None
middle_name: str | None = None  # ✅ OK: None lub str

# Wymagane z domyślną wartością
country: str = "Poland"  # ✅ OK: musi być str, domyślnie "Poland"

# BEZ wartości domyślnej - WYMAGANE!
name: str  # ❌ Musisz podać przy tworzeniu!
```

---

## 5. Zagnieżdżone modele

Modele mogą zawierać **inne modele** (kompozycja):

In [10]:
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str
    country: str = "Poland"

class User(BaseModel):
    name: str
    age: int
    address: Address  # Zagnieżdżony model!

# Tworzenie z zagnieżdżonym modelem
user = User(
    name="Jan",
    age=25,
    address=Address(street="Główna 1", city="Warszawa")
)

print(user)
print(f"\nAdres: {user.address.street}, {user.address.city}")

# Lub z dict - Pydantic automatycznie stworzy Address!
user2 = User(
    name="Anna",
    age=30,
    address={"street": "Nowa 5", "city": "Kraków"}
)

print(f"\nuser2.address type: {type(user2.address)}")
print(user2.address)

name='Jan' age=25 address=Address(street='Główna 1', city='Warszawa', country='Poland')

Adres: Główna 1, Warszawa

user2.address type: <class '__main__.Address'>
street='Nowa 5' city='Kraków' country='Poland'


### Lista zagnieżdżonych modeli

In [11]:
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    price: float

class Order(BaseModel):
    order_id: int
    items: list[Item]  # Lista modeli Item

order = Order(
    order_id=1,
    items=[
        {"name": "Apple", "price": 2.5},
        {"name": "Banana", "price": 1.2},
        Item(name="Orange", price=3.0)  # Może być dict lub model
    ]
)

print(order)
print(f"\nPierwszy item: {order.items[0].name} - {order.items[0].price} zł")

order_id=1 items=[Item(name='Apple', price=2.5), Item(name='Banana', price=1.2), Item(name='Orange', price=3.0)]

Pierwszy item: Apple - 2.5 zł


---

## 6. Serializacja - `model_validate` i `model_dump`

**Pydantic = świetny serializer!** Łatwo konwertujesz między:
- dict → Model (parsowanie, walidacja)
- Model → dict (serializacja)
- Model → JSON (serializacja do JSON string)

Przydatne w API, zapisie do plików, komunikacji między systemami.

### `model_validate` - parsowanie dict/JSON do modelu

In [14]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

# Dane z zewnątrz (np. JSON z API)
data = {"name": "Jan", "age": 25}

# Parsowanie i walidacja
user = User.model_validate(data)
print(type(user))
print(user)

# Błędne dane - ValidationError
try:
    bad_data = {"name": "Anna", "age": "not a number"}
    User.model_validate(bad_data)
except ValidationError as e:
    print(f"\n❌ ValidationError: {e.errors()[0]['msg']}")

<class '__main__.User'>
name='Jan' age=25

❌ ValidationError: Input should be a valid integer, unable to parse string as an integer


### `model_dump` - serializacja modelu do dict

**Pydantic 2.x:** `model_dump()` (była `dict()` w v1)

In [15]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int
    is_active: bool = True

user = User(name="Jan", age=25)

# Konwersja do dict
user_dict = user.model_dump()
print(user_dict)
print(f"Type: {type(user_dict)}")

# Exclude - pomiń niektóre pola
user_dict_without_age = user.model_dump(exclude={"age"})
print(f"\nBez age: {user_dict_without_age}")

# Include - tylko wybrane pola
user_dict_only_name = user.model_dump(include={"name"})
print(f"Tylko name: {user_dict_only_name}")

{'name': 'Jan', 'age': 25, 'is_active': True}
Type: <class 'dict'>

Bez age: {'name': 'Jan', 'is_active': True}
Tylko name: {'name': 'Jan'}


### `model_dump_json` - serializacja modelu do JSON string

In [16]:
user = User(name="Jan", age=25)

# JSON string
json_str = user.model_dump_json()
print(json_str)
print(f"Type: {type(json_str)}")

# Pretty print (indent)
json_pretty = user.model_dump_json(indent=2)
print(f"\nPretty:\n{json_pretty}")

{"name":"Jan","age":25,"is_active":true}
Type: <class 'str'>

Pretty:
{
  "name": "Jan",
  "age": 25,
  "is_active": true
}


---

## 7. ValidationError

Gdy walidacja się nie powiedzie, Pydantic rzuca `ValidationError`:

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class User(BaseModel):
    name: str = Field(min_length=3)
    age: int = Field(ge=0, le=120)
    email: str

# Wiele błędów naraz
try:
    user = User(name="Jo", age=150, email="invalid")
except ValidationError as e:
    print("❌ ValidationError:")
    print(e)
    
    # Dostęp do szczegółów błędów
    print("\n--- Szczegóły błędów ---")
    for error in e.errors():
        print(f"Pole: {error['loc'][0]}")
        print(f"  Błąd: {error['msg']}")
        print(f"  Typ: {error['type']}")
        print()

### Struktura ValidationError:

```python
e.errors()  # Lista błędów
[
    {
        'loc': ('name',),           # Pole które ma błąd
        'msg': 'String should...',  # Komunikat błędu
        'type': 'string_too_short', # Typ błędu
        'ctx': {...}                # Kontekst (min/max, etc.)
    },
    ...
]
```

**W FastAPI:** ValidationError automatycznie konwertowany na HTTP 422 z opisem błędów!

---

## 8. Podsumowanie

### Najważniejsze punkty:

1. **Pydantic = walidacja w runtime** (type hints + sprawdzanie)
2. **BaseModel** - podstawa każdego modelu
3. **Field** - dodatkowe reguły walidacji (min/max, pattern, etc.)
4. **Automatyczna konwersja** - "25" → 25, dict → Model
5. **Serializacja** - łatwa konwersja Model ↔ dict ↔ JSON
6. **Zagnieżdżone modele** - kompozycja modeli
7. **model_validate** - parsowanie dict/JSON do modelu
8. **model_dump** / **model_dump_json** - serializacja modelu do dict/JSON
9. **ValidationError** - szczegółowe informacje o błędach

### Podstawowy przykład (wszystko razem):

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Address(BaseModel):
    street: str
    city: str

class User(BaseModel):
    id: int = Field(gt=0)
    name: str = Field(min_length=3, max_length=50)
    age: int = Field(ge=0, le=120)
    email: str
    address: Address
    is_active: bool = True

# ✅ Tworzenie
user = User(
    id=1,
    name="Jan Kowalski",
    age=25,
    email="jan@example.com",
    address={"street": "Główna 1", "city": "Warszawa"}
)

print(user)

# ✅ Konwersja do dict
print(f"\nDict: {user.model_dump()}")

# ✅ Konwersja do JSON
print(f"\nJSON: {user.model_dump_json(indent=2)}")

# ❌ Walidacja
try:
    bad_user = User(
        id=0,  # Błąd: musi być > 0
        name="Jo",  # Błąd: za krótkie
        age=150,  # Błąd: za duże
        email="invalid",
        address={"street": "Test", "city": "Test"}
    )
except ValidationError as e:
    print(f"\n❌ Błędy walidacji: {len(e.errors())} błędów")
    for err in e.errors():
        print(f"  - {err['loc'][0]}: {err['msg']}")

### W FastAPI:

```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class User(BaseModel):
    name: str
    age: int

@app.post("/users/")
def create_user(user: User):  # Pydantic automatycznie waliduje!
    return {"created": user.name}

# FastAPI:
# 1. Parsuje JSON z requestu
# 2. Waliduje używając User (Pydantic)
# 3. Jeśli błąd → HTTP 422 z opisem
# 4. Jeśli OK → przekazuje user do funkcji
# 5. Generuje OpenAPI docs na podstawie User
```

### Następny krok:

Używanie Pydantic w FastAPI! 🚀